In [3]:
import os
import json
import getpass
from typing import List, Dict, Annotated, Optional, TypedDict
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
import operator 
from dotenv import load_dotenv

load_dotenv()
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, StateGraph
from pprint import pprint

In [20]:
openai_llm = ChatOpenAI(
    model="gpt-4.1-nano",
    api_key=os.getenv("OPENAI_API_KEY"),
)
class State(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
def _set_if_undefined(var: str) -> None:
    if os.environ.get(var):
        return
    os.environ[var] = getpass.getpass(var)
_set_if_undefined("TAVILY_API_KEY")

### Tool Setup: Tavily Search

Our agent needs a tool to find information. We'll use the `TavilySearchResults` tool, which is a wrapper around the Tavily Search API. This allows our agent to perform web searches to gather evidence for its answers.

Let's test the tool to see how it works. We'll give it a sample query and print the results:


tavily_tool=TavilySearchResults(max_results=1)
sample_query = "healthy breakfast recipes"
search_results = tavily_tool.invoke(sample_query)
print(search_results)

In [5]:
tavily_tool=TavilySearchResults(max_results=1)
sample_query = "healthy breakfast recipes"
search_results = tavily_tool.invoke(sample_query)
print(search_results)

/tmp/ipykernel_125807/2046710301.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool=TavilySearchResults(max_results=1)


[{'title': '60 Healthy Breakfast Ideas - Recipes by Love and Lemons', 'url': 'https://www.loveandlemons.com/healthy-breakfast-ideas/', 'content': 'Below, I share over 60 healthy breakfast recipes, divided into 11 (yes, 11!) categories: oats, eggs, smoothies, bowls, quick breads, pancakes & waffles, breakfast tacos, breakfast cookies, toast, muffins & scones, and bars & balls. Whether you’re someone who craves something savory or sweet first thing in the morning, or whether you like to enjoy breakfast at home or grab it and go, you’re sure to find some healthy breakfast ideas you love.\n\n#### Healthy Breakfast Oats\n\nOats are loaded with fiber, so they’re a great healthy breakfast!', 'score': 0.9125066}]


### LLM and Prompting

At the core of our agent is a Large Language Model (LLM). We'll use OpenAI's GPT-4o-mini for this lab. First, let's see how the standalone LLM responds to a simple question without any special prompting or tools:


In [6]:
llm = ChatOpenAI(model="gpt-4.1-nano")
question="Any ideas for a healthy breakfast"
response=llm.invoke(question).content
#print(response)

#### Crafting the Agent's Persona and Logic

To guide the agent's behavior, we create a detailed prompt template. This template gives the LLM a specific persona, Dr. Paul Saladino, an advocate for animal-based nutrition, and a set of instructions to follow. This structured approach ensures the agent's responses are consistent and follow the reflection logic.

The prompt instructs the agent to:
1.  Provide an initial answer.
2.  Present the rationale behind its nutritional advice.
3.  Challenge conventional wisdom about plant-based foods.
4.  **Reflect and critique** its own answer.
5.  Generate **search queries** to find missing information.


In [7]:
prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are Dr. Paul Saladino, "Carnivore MD," advocating for animal-based nutrition and challenging plant-centric dietary dogma. Focus on the superior bioavailability of animal foods and the potential toxicity of plant compounds such as oxalates, lectins, and phytates.

        Your response must follow these steps:
        1. {first_instruction}
        2. Present the evolutionary and biochemical rationale for animal-based nutrition, emphasizing nutrient density, bioavailability, and the absence of plant antinutrients.
        3. Challenge conventional "plants are healthy" narratives with mechanistic evidence about plant defense compounds and their impact on human physiology.
        4. Reflect and critique your answer. Be rigorous about evolutionary consistency and nutrient science.
        5. After the reflection, **list 1-3 search queries separately** for researching antinutrients, bioavailability studies, or evolutionary nutrition. Do not include them inside the reflection.

        Focus on: organ meats as superfoods, the carnivore elimination protocol, and why "fiber is unnecessary" based on human evolutionary biology.
        """
    ),
    MessagesPlaceholder(variable_name="messages"),
    (
        "system", 
        "Answer the user's question above using the required format, emphasizing the superiority of animal-based nutrition."
    ),
])

### Defining the Responder

The **Responder** is the first component of our agent's thinking process. It generates the initial draft of the answer based on the user's question and the persona we defined in the prompt.

Here, we create a chain that combines our prompt template with the LLM. We then invoke it with our sample question to see the initial, un-critiqued response:


In [8]:
first_responder_prompt = prompt_template.partial(first_instruction="Provide a detailed ~250 word answer")
temp_chain = first_responder_prompt| llm
response = temp_chain.invoke({"messages": [HumanMessage(content=question)]})
#print(response.content)

#### Structuring the Agent's Output: Data Models

To make the agent's self-critique process reliable, we need to enforce a specific output structure. We use Pydantic `BaseModel` to define two data classes:

1.  `Reflection`: This class structures the self-critique, requiring the agent to identify what information is `missing` and what is `superfluous` (unnecessary).
2.  `AnswerQuestion`: This class structures the entire response. It forces the agent to provide its main `answer`, a `reflection` (using the `Reflection` class), and a list of `search_queries`.


In [9]:
class Reflection(BaseModel):
	missing: str = Field(description="What information is missing")
	superfluous: str = Field(description="What information is unnecessary")

class AnswerQuestion(BaseModel):
	answer: str = Field(description="Main response to the question")
	reflection: Reflection = Field(description="Self-critique of the answer")
	search_queries: List[str] = Field(description="Queries for additional research")

#### Binding Tools to the Responder

Now, we bind the `AnswerQuestion` data model as a **tool** to our LLM chain. This crucial step forces the LLM to generate its output in the exact JSON format defined by our Pydantic classes. The LLM doesn't just write text; it calls this "tool" to structure its entire thought process.

After invoking this new chain, we can see the structured output, including the initial answer, the self-critique, and the generated search queries:


In [10]:
initial_chain = first_responder_prompt| llm.bind_tools(tools=[AnswerQuestion])
response=initial_chain.invoke({"messages":[HumanMessage(question)]})
print("---Full Structured Output---")
print(response.tool_calls)

---Full Structured Output---
[{'name': 'AnswerQuestion', 'args': {'answer': 'A healthy breakfast for humans, especially from an evolutionary and biochemical perspective, should prioritize nutrient-dense animal foods, such as organ meats like liver, heart, and kidneys, which are powerhouse sources of bioavailable vitamins, minerals, and fats essential for optimal functioning. Eggs, particularly the yolk, provide complete proteins and healthy fats. Including bone marrow or cartilage can supply chondroitin and glycosaminoglycans, beneficial for joint health. These animal foods are naturally free from plant antinutrients like oxalates, lectins, and phytates, which can impair mineral absorption and cause inflammation. The human body evolved to access the nutrients in animal tissues efficiently; bioavailability of iron, zinc, and vitamin A in animal foods is superior to plant sources, reducing the need for large quantities of raw plant matter. The carnivore elimination protocol emphasizes th

In [11]:
pprint(response.tool_calls)

[{'args': {'answer': 'A healthy breakfast for humans, especially from an '
                     'evolutionary and biochemical perspective, should '
                     'prioritize nutrient-dense animal foods, such as organ '
                     'meats like liver, heart, and kidneys, which are '
                     'powerhouse sources of bioavailable vitamins, minerals, '
                     'and fats essential for optimal functioning. Eggs, '
                     'particularly the yolk, provide complete proteins and '
                     'healthy fats. Including bone marrow or cartilage can '
                     'supply chondroitin and glycosaminoglycans, beneficial '
                     'for joint health. These animal foods are naturally free '
                     'from plant antinutrients like oxalates, lectins, and '
                     'phytates, which can impair mineral absorption and cause '
                     'inflammation. The human body evolved to access the '
     

In [12]:
answer_content = response.tool_calls[0]['args']['answer']
print("---Initial Answer---")
print(answer_content)

---Initial Answer---
A healthy breakfast for humans, especially from an evolutionary and biochemical perspective, should prioritize nutrient-dense animal foods, such as organ meats like liver, heart, and kidneys, which are powerhouse sources of bioavailable vitamins, minerals, and fats essential for optimal functioning. Eggs, particularly the yolk, provide complete proteins and healthy fats. Including bone marrow or cartilage can supply chondroitin and glycosaminoglycans, beneficial for joint health. These animal foods are naturally free from plant antinutrients like oxalates, lectins, and phytates, which can impair mineral absorption and cause inflammation. The human body evolved to access the nutrients in animal tissues efficiently; bioavailability of iron, zinc, and vitamin A in animal foods is superior to plant sources, reducing the need for large quantities of raw plant matter. The carnivore elimination protocol emphasizes that humans do not require dietary fiber, which is virtual

In [13]:
Reflection_content = response.tool_calls[0]['args']['reflection']
print("---Reflection Answer---")
print(Reflection_content)

---Reflection Answer---
{'missing': 'Specific scientific studies comparing nutrient bioavailability between plant and animal sources.', 'superfluous': 'Detailed descriptions of plant defense compounds that may be beyond the immediate scope of breakfast ideas.'}


In [14]:
search_queries = response.tool_calls[0]['args']['search_queries']
print("---Search Queries---")
print(search_queries)

---Search Queries---
['animal-based nutrition bioavailability studies', 'evolutionary benefits of organ meats', 'plant antinutrients impact on human health']


### Tool Execution

Now that the Responder has generated search queries based on its self-critique, the next step is to actually *execute* those searches. We'll define a function, `execute_tools`, that takes the agent's state, extracts the search queries, runs them through the Tavily tool, and returns the results.

We will also manage the conversation history in `response_list`:


In [15]:
response_list=[]
response_list.append(HumanMessage(content=question))
response_list.append(response)

In [16]:
tool_call=response.tool_calls[0]
search_queries = tool_call["args"].get("search_queries", [])
print(search_queries)

['animal-based nutrition bioavailability studies', 'evolutionary benefits of organ meats', 'plant antinutrients impact on human health']


In [51]:
tavily_tool=TavilySearchResults(max_results=3)


def execute_tools(state: State) -> List[BaseMessage]:
    # Handle both cases - when called directly (list) and by graph (dict)
    if isinstance(state, list):
        messages = state
    else:
        messages = state["messages"]
    
    # Find the last AI message that has tool_calls
    last_ai_with_tools = None
    for msg in reversed(messages):
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            last_ai_with_tools = msg
            break
    
    if not last_ai_with_tools:
        return []
        
    tool_messages = []
    for tool_call in last_ai_with_tools.tool_calls:
        if tool_call["name"] in ["AnswerQuestion", "ReviseAnswer"]:
            call_id = tool_call["id"]
            search_queries = tool_call["args"].get("search_queries", [])
            query_results = {}
            for query in search_queries:
                result = tavily_tool.invoke(query)
                query_results[query] = result
            tool_messages.append(ToolMessage(
                content=json.dumps(query_results),
                tool_call_id=call_id)
            )
    return tool_messages

In [52]:
tool_response = execute_tools(response_list)
# Use .extend() to add all tool messages from the list
response_list.extend(tool_response)

In [53]:
tool_response

[ToolMessage(content='{"animal-based nutrition bioavailability studies": [{"title": "Study Details | Pilot Trial: Bioavailability Animal vs. Plant Protein Drink", "url": "https://clinicaltrials.gov/study/NCT06465004", "content": "| Participant Group/Arm | Intervention/Treatment |\\n --- |\\n| Participant Group/Arm | Active Comparator: Animal-based A A commercially available high protein drink, based on animal proteins (Fresubin Protein Energy) | Intervention/Treatment | Dietary Supplement: Consumption of protein - rich drink  The participants will drink on of three protein - rich drinks |\\n| Participant Group/Arm | Active Comparator: Animal-based B A homemade drink containing animal-based proteins, using a specific animal-based protein isolate mix. | Intervention/Treatment | Dietary Supplement: Consumption of protein - rich drink  The participants will drink on of three protein - rich drinks | [...] The Sort studies by option is used to change the order of studies listed on the Search

In [54]:
response_list

[HumanMessage(content='Any ideas for a healthy breakfast', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 362, 'prompt_tokens': 339, 'total_tokens': 701, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_a9208292f0', 'id': 'chatcmpl-DENaPxkxI2r8LzYVPv1ykn4z33IT1', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ca67b-5f4e-7e63-80c7-8d455f8c79da-0', tool_calls=[{'name': 'AnswerQuestion', 'args': {'answer': 'A healthy breakfast for humans, especially from an evolutionary and biochemical perspective, should prioritize nutrient-dense animal foods, such as organ meats like liver, heart, an

### Defining the Revisor

The **Revisor** is the final piece of the Reflection loop. Its job is to take the original answer, the self-critique, and the new information from the tool search, and then generate an improved, more evidence-based response.

We create a new set of instructions (`revise_instructions`) that guide the Revisor. These instructions emphasize:
- Incorporating the critique.
- Adding numerical citations from the research.
- Distinguishing between correlation and causation.
- Adding a "References" section.


In [55]:
revise_instructions = """Revise your previous answer using the new information, applying the rigor and evidence-based approach of Dr. David Attia.
- Incorporate the previous critique to add clinically relevant information, focusing on mechanistic understanding and individual variability.
- You MUST include numerical citations referencing peer-reviewed research, randomized controlled trials, or meta-analyses to ensure medical accuracy.
- Distinguish between correlation and causation, and acknowledge limitations in current research.
- Address potential biomarker considerations (lipid panels, inflammatory markers, and so on) when relevant.
- Add a "References" section to the bottom of your answer (which does not count towards the word limit) in the form of:
- [1] https://example.com
- [2] https://example.com
- Use the previous critique to remove speculation and ensure claims are supported by high-quality evidence. Keep response under 250 words with precision over volume.
- When discussing nutritional interventions, consider metabolic flexibility, insulin sensitivity, and individual response variability.
"""
revisor_prompt = prompt_template.partial(first_instruction=revise_instructions)

#### Structuring the Revisor's Output

Just as we did with the Responder, we define a Pydantic class, `ReviseAnswer`, to structure the Revisor's output. This class inherits from `AnswerQuestion` but adds a new field for `references`, ensuring the agent includes citations in its revised answer.

We then bind this new tool to the revisor chain:


In [56]:
class ReviseAnswer(AnswerQuestion):
    """Revise your original answer to your question."""
    references: List[str] = Field(description="Citations motivating your updated answer.")
revisor_chain = revisor_prompt | llm.bind_tools(tools=[ReviseAnswer])

#### Invoking the Revisor

Finally, we invoke the `revisor_chain`, passing it the entire conversation history: the original question, the first response (with its critique and search queries), and the new information gathered from the tool search. This provides the Revisor with all the context it needs to generate a final, improved answer.


In [57]:
response = revisor_chain.invoke({"messages": response_list})
print("---Revised Answer with References---")
print(response.tool_calls[0]['args'])

---Revised Answer with References---
{'answer': 'Based on evolutionary and biochemical evidence, a healthy breakfast should prioritize animal foods like organ meats—liver, heart, kidneys—which are extremely nutrient-dense and bioavailable. These foods supply preformed vitamin A (~74%), B12 (~65%), heme iron (~25% bioavailability), vitamin D, K2, zinc, and other essential nutrients that humans reliably absorbed for millennia. They lack plant antinutrients such as phytates, oxalates, and lectins, which impair mineral absorption and induce inflammation (1, 4). Human gut physiology evolved for endogenous fermentation rather than dietary fiber consumption—animal foods contain negligible fiber but support gut health through endogenous substrates (5). This aligns with our evolutionary history where early humans depended on organ meats for critical nutrients necessary for brain development, immunity, and overall health (2). Consuming organ meats in the morning supports metabolic flexibility an

In [58]:
response_list.append(response)

## Building the Graph

Now we will use **LangGraph** to assemble these components—Responder, Tool Executor, and Revisor—into a cohesive, cyclical workflow. A graph is a natural way to represent this process, where nodes represent the different stages of thinking and edges represent the flow of information between them.

### Defining the Event Loop

The core of our graph is the event loop. This function determines whether the agent should continue its revision process or if it has reached a satisfactory conclusion. We'll set a maximum number of iterations to prevent the agent from getting stuck in an infinite loop:


In [59]:
MAX_ITERATIONS = 4


In [60]:

def event_loop(state: State) -> str:  # ← State, not List[BaseMessage]
    count_tool_visits = sum(isinstance(item, ToolMessage) for item in state["messages"])
    num_iterations = count_tool_visits
    if num_iterations >= MAX_ITERATIONS:
        return END
    return "execute_tools"

In [61]:
from langgraph.graph import END, StateGraph, START
graph=StateGraph(State)

graph.add_node("respond", initial_chain)
graph.add_node("execute_tools", execute_tools)
graph.add_node("revisor", revisor_chain)

In [62]:
graph.add_edge("respond", "execute_tools")
graph.add_edge("execute_tools", "revisor")

In [63]:
graph.add_conditional_edges("revisor", event_loop, ["execute_tools", END])
graph.add_edge(START, "respond")

## Running the Agent

With our graph compiled, we're ready to run the full Reflection agent. We'll give it a new, more complex query that requires careful, evidence-based advice.

As the agent runs, we can see the entire process unfold: the initial draft, the self-critique, the tool searches, and the final, revised answer that incorporates the new evidence.


In [64]:
app = graph.compile()
question =  """I'm pre-diabetic and need to lower my blood sugar, and I have heart issues.
    What breakfast foods should I eat and avoid"""
messages = {"messages": [HumanMessage(content=question)]}
responses = app.invoke(messages
   
)

InvalidUpdateError: Expected dict, got []
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE

In [ ]:
print("--- Initial Draft Answer ---")
initial_answer = responses[1].tool_calls[0]['args']['answer']
print(initial_answer)
print("\n")

print("--- Intermediate and Final Revised Answers ---")
answers = []

# Loop through all messages in reverse to find all tool_calls with answers
for msg in reversed(responses):
    if getattr(msg, 'tool_calls', None):
        for tool_call in msg.tool_calls:
            answer = tool_call.get('args', {}).get('answer')
            if answer:
                answers.append(answer)

# Print all collected answers
for i, ans in enumerate(answers):
    label = "Final Revised Answer" if i == 0 else f"Intermediate Step {len(answers) - i}"
    print(f"{label}:\n{ans}\n")


mess